In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
import os
import re
import os

TITLES = [
    "ZZx1",  # Train
    "ZZx2",  # Val
    "ZZxReto", # test
]

PREDICTORS = ["e_a_d", "e_a_e", "Se_a", "De_a", "costheta"]   
TARGET_INT = ["x"]  
TARGET = ["dx"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
     
TIME_STEPS = 6
TS = 0.07
PLOT = False

In [2]:
Datasets = []
NormDatasets = []

# Ler Datasets.xlsx
for title in TITLES:
    df = pd.read_excel("./Data/Datasets.xlsx", sheet_name=title)
    Datasets.append(df)

# Ler NormDatasets.xlsx
for title in TITLES:
    df = pd.read_excel("./Data/NormDatasets.xlsx", sheet_name=title)
    NormDatasets.append(df)
    
SCALER = joblib.load("./scalers/scaler.pkl")    
OUT_SCALER = joblib.load("./scalers/out_scaler.pkl")

In [3]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)


In [4]:
R = tf.constant(0.0328, dtype=tf.float32)
L = tf.constant(0.0615, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(phi_d, phi_e, theta):

    #dtheta_cin = (R / (2 * L)) * (phi_d - phi_e)
    dx_cin = (R / 2) * tf.cos(theta) * (phi_d + phi_e)
    # dy_cin = (R / 2) * tf.sin(theta) * (phi_d + phi_e)

    # return [dtheta_cin, dx_cin, dy_cin]
    return [dx_cin]


def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["phi_d"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["phi_e"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T, np.vstack(dq).T

In [5]:
def to_scalar(x):
    return float(x[0]) if isinstance(x, list) else float(x)

In [8]:

# Regex para extrair os parâmetros do nome do arquivo
# ex: model_arch36-22_r0.9_Ld0.3_Lp0.7_seed5124.keras
PATTERN = re.compile(
    r"^model_arch(?P<arch>[\d\-]+)_r(?P<r>-?[\d.]+)_Ld(?P<Ld>-?[\d.]+)_Lp(?P<Lp>-?[\d.]+)_seed(?P<seed>\d+)\.keras$"
)

def PlotOut(ax, title, target_name, y_true, y_pred, y_cin):
    time = (np.arange(len(y_pred)).astype(float) * 0.07).round(5)

    ax.plot(time, y_true, '-', linewidth=1.5, label='Amostras Reais')
    ax.plot(time, y_pred, '--', linewidth=1.5, label='Valores preditos')
    ax.plot(time, y_cin, ':', linewidth=2, label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    from sklearn.metrics import r2_score, mean_squared_error

    n_targets = len(TARGET)
    n_datasets = len(Datasets)

    if PLOT:
        fig, axs = plt.subplots(
            n_datasets,
            2 * n_targets,
            figsize=(6 * 2 * n_targets, 4 * n_datasets)
        )
        axs = np.atleast_2d(axs)

    metrics = {name: {} for name in TARGET_INT}

    for i, NormDataset in enumerate(NormDatasets):

        x = NormDataset[PREDICTORS]
        y = Datasets[i][TARGET_INT]
        dy_true = Datasets[i][TARGET].values

        x, y = CreateSequences(x, y, TIME_STEPS)

        # alinhar derivada
        dy_true = dy_true[TIME_STEPS:]

        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()
        dy_pred = OUT_SCALER.inverse_transform(pred)

        y_true = y.copy()
        y_pred = np.zeros_like(dy_pred)

        dy_cin, y_cin = GetCin(Datasets[i])
        y_cin = y_cin[:y_true.shape[0]]
        dy_cin = dy_cin[:dy_pred.shape[0]]

        init_vals = np.array([Datasets[i][name].iloc[0] for name in TARGET_INT])

        for j in range(n_targets):
            y_pred[:, j] = init_vals[j] + np.cumsum(dy_pred[:, j] * TS)

        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = r2_score(dy_true[:, j], dy_pred[:, j])

            key_r2 = f"R2_{TITLES[i]}"
            key_mse = f"MSE_{TITLES[i]}"

            metrics[name].setdefault(key_r2, []).append(r2)
            metrics[name].setdefault(key_mse, []).append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, R² diff = {mse:.4f}")

            if PLOT:
                ax_y = axs[i, j]
                PlotOut(ax_y, TITLES[i], name,
                        y_true[:, j], y_pred[:, j], y_cin[:, j])

                ax_dy = axs[i, j + n_targets]
                PlotOut(ax_dy, TITLES[i], f"d{name}",
                        dy_true[:, j], dy_pred[:, j], dy_cin[:, j])

    if PLOT:
        plt.tight_layout()

    return metrics

def UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file):

    model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_Ld{Ld}_Lp{Lp}_seed{seed}"

    row = {
        "model": model_name,
        "Neurons": arch,
        "Ld": Ld,
        "Lp": Lp,
        "reg": r,
        "seed": seed,
    }

    for name in TARGET_INT:
        entry = {}
        for title in TITLES:
            safe_title = title.replace("-", "_")  
            entry[f"R2_{safe_title}_{name}"] = to_scalar(metrics[name][f"R2_{title}"])
            entry[f"MSE_{safe_title}_{name}"] = to_scalar(metrics[name][f"MSE_{title}"])
        row.update(entry)
        df = pd.DataFrame([row])

    try:
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        df.to_excel(excel_file, index=False)

    print(f"Modelo {arch} | Ld={Ld} Lp={Lp} r={r} seed={seed} salvo.")


def ParseModelFilename(filename):
    m = PATTERN.match(filename)
    if not m:
        return None

    arch = [int(x) for x in m.group("arch").split("-")]
    r    = float(m.group("r"))
    Ld   = float(m.group("Ld"))
    Lp   = float(m.group("Lp"))
    seed = int(m.group("seed"))

    return arch, r, Ld, Lp, seed


def RebuildResultsNL(n_layers, models_dir="models", excel_file=None, verbose=True):
    excel_file = excel_file or f"resultados-{n_layers}l.xlsx"
    if os.path.exists(excel_file):
        os.remove(excel_file)

    all_files = sorted(f for f in os.listdir(models_dir) if f.endswith(".keras"))

    n_found = 0
    for filename in all_files:
        parsed = ParseModelFilename(filename)
        if parsed is None:
            continue

        arch, r, Ld, Lp, seed = parsed
        if len(arch) != n_layers:
            continue

        n_found += 1
        model = tf.keras.saving.load_model(os.path.join(models_dir, filename))
        metrics = EvalModel(model)
        UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file)

        if verbose:
            print(f"[{n_found}] {filename} -> ok")

    print(f"\nConcluído. {n_found} modelos de {n_layers} camadas -> {excel_file}")


PLOT = False
RebuildResultsNL(1)   # gera resultados-3l.xlsx

x | ZZx1 -> R² = 0.9260, R² diff = 0.9513
x | ZZx2 -> R² = 0.8397, R² diff = 0.8637
x | ZZxReto -> R² = 0.7312, R² diff = 0.9272
Modelo [100] | Ld=0.3 Lp=0.7 r=0.01 seed=1457 salvo.
[1] model_arch100_r0.01_Ld0.3_Lp0.7_seed1457.keras -> ok
x | ZZx1 -> R² = 0.8933, R² diff = 0.9464
x | ZZx2 -> R² = 0.7578, R² diff = 0.8201
x | ZZxReto -> R² = 0.8372, R² diff = 0.9301
Modelo [100] | Ld=0.3 Lp=0.7 r=0.01 seed=1771 salvo.
[2] model_arch100_r0.01_Ld0.3_Lp0.7_seed1771.keras -> ok
x | ZZx1 -> R² = 0.9101, R² diff = 0.9424
x | ZZx2 -> R² = 0.9415, R² diff = 0.8643
x | ZZxReto -> R² = 0.7334, R² diff = 0.9134
Modelo [100] | Ld=0.3 Lp=0.7 r=0.01 seed=3191 salvo.
[3] model_arch100_r0.01_Ld0.3_Lp0.7_seed3191.keras -> ok
x | ZZx1 -> R² = 0.9405, R² diff = 0.9569
x | ZZx2 -> R² = 0.1961, R² diff = 0.8281
x | ZZxReto -> R² = 0.7111, R² diff = 0.9301
Modelo [100] | Ld=0.3 Lp=0.7 r=0.01 seed=3379 salvo.
[4] model_arch100_r0.01_Ld0.3_Lp0.7_seed3379.keras -> ok
x | ZZx1 -> R² = 0.9170, R² diff = 0.9448
x 